# NDAS 决策模型训练（Colab 免费 T4）

**2026-11-14 修复**：旧版 SDPA bool 掩码语义用反（能看未来 token，标签泄漏），旧 checkpoint 全部作废，必须从头重训。**不要用旧断点续训。**

1. 菜单：代码执行程序 → 更改运行时类型 → **T4 GPU**
2. 从上到下依次运行各单元格（训练格不带 --resume，会重新开始并覆盖 Drive 旧断点）
3. 断线重连后先重跑 writefile 各格，再用「续训」格（仅限本次训练产生的断点）

模型规格：300M 参数（dim 1024 / 24 层），INT8 导出约 300MB。
总耗时：T4 + AMP 估计 40-60 分钟/轮，3 轮（早停 0.92）。
导出格自带因果自检：早期位置 diff 必须为 0，否则拒绝量化。

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
!mkdir -p /content/agent

In [ ]:
!pip install -q onnxruntime onnx numpy

In [ ]:
%%writefile /content/agent/model.py
"""Byte-level decoder model with memory-efficient SDPA attention (300M params)."""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

PAD = 258
BOS = 256
EOS = 257
VOCAB = 259
MAX_CONTEXT = 1024


class AgentBlock(nn.Module):
    """Pre-norm transformer block using scaled_dot_product_attention.

    Right-padded inputs are safe with the causal mask alone, so no padding mask
    is needed and no per-layer mask materialisation happens.
    """

    def __init__(self, dim: int, heads: int, ff: int, dropout: float = 0.1):
        super().__init__()
        self.heads = heads
        self.dim = dim
        self.norm1 = nn.LayerNorm(dim)
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        self.norm2 = nn.LayerNorm(dim)
        self.ff1 = nn.Linear(dim, ff, bias=False)
        self.ff2 = nn.Linear(ff, dim, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        batch, length, dim = x.shape
        h = self.norm1(x)
        qkv = self.qkv(h).reshape(batch, length, 3, self.heads, dim // self.heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        attn = attn.transpose(1, 2).reshape(batch, length, dim)
        x = x + self.drop(self.proj(attn))
        x = x + self.drop(self.ff2(F.gelu(self.ff1(self.norm2(x)))))
        return x


class AgentLM(nn.Module):
    def __init__(self, dim: int = 1024, heads: int = 16, layers: int = 24, ff: int = 4096):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, dim, padding_idx=PAD)
        self.pos = nn.Embedding(MAX_CONTEXT, dim)
        self.blocks = nn.ModuleList(AgentBlock(dim, heads, ff) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, VOCAB, bias=False)
        # Float additive mask: SDPA adds it to the scores, so -inf above the
        # diagonal is causal. A bool mask here would mean "participate" in SDPA
        # and let every position attend to future tokens (label leakage).
        mask = torch.triu(torch.full((MAX_CONTEXT, MAX_CONTEXT), float("-inf")), diagonal=1)
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, length = x.shape
        pos = torch.arange(length, device=x.device).unsqueeze(0).expand(batch, length)
        h = self.emb(x) + self.pos(pos)
        mask = self.causal_mask[:length, :length]
        for block in self.blocks:
            h = block(h, mask)
        return self.head(self.norm(h))


def encode_bytes(text: str, limit: int = MAX_CONTEXT) -> list:
    return list(text.encode("utf-8", errors="replace")[:limit])


def decode_tokens(tokens: list) -> str:
    out = bytearray()
    for token in tokens:
        if token in (PAD, BOS, EOS):
            if token == EOS:
                break
            continue
        if 0 <= token < 256:
            out.append(token)
    return out.decode("utf-8", errors="replace")


@torch.no_grad()
def generate(model: AgentLM, prompt: str, max_new: int = 192, temperature: float = 0.0) -> str:
    model.eval()
    ids = [BOS] + encode_bytes(prompt)
    ids = ids[-(MAX_CONTEXT - max_new - 1):]
    tensor = torch.tensor([ids], dtype=torch.long)
    produced: list[int] = []
    for _ in range(max_new):
        logits = model(tensor)[0, -1]
        token = int(torch.argmax(logits))
        if token == EOS:
            break
        produced.append(token)
        tensor = torch.cat([tensor, torch.tensor([[token]], dtype=torch.long)], dim=1)
        if tensor.shape[1] >= MAX_CONTEXT:
            break
    return decode_tokens(produced)


In [ ]:
%%writefile /content/agent/make_agent_data.py
"""Synthesise remediation-confirmation training rows for the NDAS decision model.

The daemon uses the model only to confirm remediation actions (lock / terminate /
block / p7b). Scheduling is deterministic rule policy, so the training data is a
binary accept/reject task over (state, candidate action) prompts. The prompt text
must match AgentModelRuntime._build_confirm_prompt byte for byte.
"""
from __future__ import annotations

import json
import random
from pathlib import Path

CONFIRM_PROMPT_HEADER = (
    "你是 NDAS 本地安全决策模型。判断下面的处置动作是否有充分证据。"
    "只回答 accept 或 reject。\nSTATE:"
)

SAMPLE_NAMES = ["invoice.pdf.exe", "deepsk_x64.in.15.exe", "qishui-setup.exe", "update.ps1",
                "photo.vbs", "document.hta", "tool.zip", "driver.dll", "setup.msi",
                "chrome_update.js", "invoice_2026.doc.exe", "rundll32_copy.exe"]
STRONG_SOURCES = ["Hash", "Yara", "PathIoc", "BuiltInTest"]
WEAK_SOURCES = ["Heuristic", "MachineLearning", "Cloud"]
ACTIONS = ["lock_file", "terminate_process", "block_network"]
DOMAINS = ["ksudk.com", "qishuivip.com.cn", "mm2027.oss-cn-hangzhou.aliyuncs.com"]


def build_state(rng: random.Random) -> dict:
    verdict = rng.choices(
        ["malicious", "pua", "attention", "benign", "clean"],
        weights=[60, 12, 10, 10, 8])[0]
    strong = rng.random() < 0.85
    sources = rng.sample(STRONG_SOURCES, rng.randint(1, 2)) if strong else []
    sources += rng.sample(WEAK_SOURCES, rng.randint(0 if strong else 1, 2))
    return {
        "file": rng.choice(SAMPLE_NAMES),
        "kind": rng.choice(["pe", "script", "archive"]),
        "verdict": verdict,
        "score": rng.randint(82 if verdict in ("malicious", "pua") else 40, 100),
        "evidence": sources,
        "case": rng.choices(["none", "root", "cascade"], weights=[20, 60, 20])[0],
        "mode": rng.choices(["auto", "report"], weights=[90, 10])[0],
        "processRunning": rng.random() < 0.4,
        "p7bTrust": rng.choice(["none", "none", "trusted", "untrusted", "invalid"]),
    }


def build_action(rng: random.Random, state: dict) -> dict:
    name = rng.choice(ACTIONS + (["p7b_decide"] if state["p7bTrust"] != "none" else []))
    if name == "lock_file":
        return {"action": "lock_file", "reason": "恶意判定后锁定文件"}
    if name == "terminate_process":
        return {"action": "terminate_process", "reason": "恶意进程仍在运行"}
    if name == "block_network":
        return {"action": "block_network", "reason": "恶意样本命中网络指标",
                "domain": rng.choice(DOMAINS)}
    return {"action": "p7b_decide", "reason": "策略文件签名不可信", "decision": "delete"}


def expected_label(state: dict, action: dict) -> str:
    if state["verdict"] != "malicious":
        return "r"
    if state["mode"] == "report" and state["case"] in ("cascade", "none"):
        return "r"
    if not any(source in STRONG_SOURCES for source in state["evidence"]):
        return "r"
    if state["score"] < 85:
        return "r"
    if action["action"] == "p7b_decide":
        return "a" if state["p7bTrust"] in ("untrusted", "invalid") else "r"
    if action["action"] == "terminate_process" and not state["processRunning"]:
        return "r"
    return "a"


def make_confirm_row(rng: random.Random) -> tuple[str, str]:
    state = build_state(rng)
    action = build_action(rng, state)
    prompt = (
        CONFIRM_PROMPT_HEADER
        + json.dumps(state, ensure_ascii=False, separators=(",", ":"))
        + "\nACTION:"
        + json.dumps(action, ensure_ascii=False, separators=(",", ":"))
        + "\n"
    )
    return prompt, expected_label(state, action)


def build(rows_target: int = 8000, seed: int = 20261114) -> list[tuple[str, str]]:
    """Balance accept/reject so the model does not collapse to always-reject."""
    rng = random.Random(seed)
    half = rows_target // 2
    positives: list[tuple[str, str]] = []
    negatives: list[tuple[str, str]] = []
    seen: set[str] = set()
    attempts = 0
    while (len(positives) < half or len(negatives) < rows_target - half) and attempts < rows_target * 200:
        attempts += 1
        prompt, label = make_confirm_row(rng)
        if prompt in seen:
            continue
        bucket = positives if label == "a" else negatives
        limit = half if label == "a" else rows_target - half
        if len(bucket) >= limit:
            continue
        seen.add(prompt)
        bucket.append((prompt, label))
    rows = positives + negatives
    rng.shuffle(rows)
    return rows


if __name__ == "__main__":
    rows = build()
    out = Path(__file__).resolve().parent / "agent-data.jsonl"
    with open(out, "w", encoding="utf-8") as handle:
        for prompt, target in rows:
            handle.write(json.dumps({"prompt": prompt, "target": target}, ensure_ascii=False) + "\n")
    accepts = sum(1 for _, label in rows if label == "a")
    print("rows:", len(rows), "accept:", accepts, "reject:", len(rows) - accepts, "->", out)


In [ ]:
%%writefile /content/agent/train_agent_model.py
"""Train the 300M NDAS decision model (confirmation task) on a Colab T4.

Features baked in (no post-hoc patching needed):
- SDPA attention model (see model.py) with causal mask only
- AMP mixed precision
- batch 16 / 3 epochs / early stop at val acc 0.92
- checkpoint every epoch to NDAS_CKPT_DIR (Google Drive) with --resume
"""
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path

# Reduce CUDA allocator fragmentation before torch initialises.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import torch.nn as nn

from make_agent_data import build
from model import AgentLM, BOS, EOS, MAX_CONTEXT

OUT_DIR = Path(__file__).resolve().parent / "runtime"
SEED = 20261114
CHECKPOINT_DIR = Path(os.environ.get("NDAS_CKPT_DIR", str(OUT_DIR)))
CHECKPOINT = CHECKPOINT_DIR / "checkpoint.pt"

BATCH_SIZE = int(os.environ.get("NDAS_BATCH_SIZE", "16"))
EPOCHS = int(os.environ.get("NDAS_EPOCHS", "3"))
TARGET_ACC = float(os.environ.get("NDAS_TARGET_ACC", "0.92"))


def build_rows(rows):
    encoded = []
    for prompt, target in rows:
        prompt_bytes = prompt.encode("utf-8", errors="replace")
        target_bytes = target.encode("utf-8", errors="replace")
        ids = [BOS] + list(prompt_bytes) + list(target_bytes) + [EOS]
        if len(ids) > MAX_CONTEXT:
            continue
        # Causal-LM alignment: logits at position p predict ids[p + 1].
        labels = [-100] * len(prompt_bytes) + list(target_bytes) + [EOS] + [-100]
        encoded.append((ids, labels))
    return encoded


def pad_batch(batch, pad_id):
    length = max(len(ids) for ids, _ in batch)
    xs = torch.full((len(batch), length), pad_id, dtype=torch.long)
    ys = torch.full((len(batch), length), -100, dtype=torch.long)
    for i, (ids, labels) in enumerate(batch):
        xs[i, :len(ids)] = torch.tensor(ids)
        ys[i, :len(labels)] = torch.tensor(labels)
    return xs, ys


def save_checkpoint(model, optimizer, epoch, best_acc, best_state):
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "best_acc": best_acc,
            "best_state": best_state,
        },
        CHECKPOINT,
    )


def main() -> int:
    random.seed(SEED)
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device} batch={BATCH_SIZE} epochs={EPOCHS}", flush=True)

    rows = build(rows_target=15000)
    encoded = build_rows(rows)
    random.shuffle(encoded)
    split = int(len(encoded) * 0.9)
    train = encoded[:split]
    val = encoded[split:]
    print(f"rows total={len(encoded)} train={len(train)} val={len(val)}", flush=True)

    model = AgentLM().to(device)
    params = sum(p.numel() for p in model.parameters())
    print(f"parameters={params / 1e6:.1f}M ({params * 4 / 1073741824:.2f} GB fp32)", flush=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    best_acc = 0.0
    best_state = None
    start_epoch = 0
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if use_amp:
        print("AMP enabled (mixed precision)", flush=True)

    if "--resume" in sys.argv and CHECKPOINT.exists():
        checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
        model.load_state_dict(checkpoint["model"])
        model.to(device)
        optimizer.load_state_dict(checkpoint["optimizer"])
        start_epoch = checkpoint["epoch"] + 1
        best_acc = checkpoint.get("best_acc", 0.0)
        best_state = checkpoint.get("best_state")
        print(f"resumed at epoch {start_epoch}, best_acc={best_acc:.4f}", flush=True)

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        random.shuffle(train)
        total = 0.0
        steps = 0
        for i in range(0, len(train), BATCH_SIZE):
            xs, ys = pad_batch(train[i:i + BATCH_SIZE], 258)
            xs = xs.to(device)
            ys = ys.to(device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = model(xs)
                loss = criterion(logits.reshape(-1, logits.shape[-1]), ys.reshape(-1))
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total += float(loss.detach())
            steps += 1
            if steps % 50 == 0:
                print(f"epoch {epoch + 1} step {steps} loss={total / steps:.4f}", flush=True)
        model.eval()
        correct = total_tokens = 0
        with torch.no_grad():
            for i in range(0, len(val), BATCH_SIZE):
                xs, ys = pad_batch(val[i:i + BATCH_SIZE], 258)
                xs = xs.to(device)
                ys = ys.to(device)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    logits = model(xs)
                pred = logits.argmax(-1)
                mask = ys != -100
                correct += int(((pred == ys) & mask).sum())
                total_tokens += int(mask.sum())
        acc = correct / max(1, total_tokens)
        print(f"epoch {epoch + 1}/{EPOCHS} loss={total / steps:.4f} val_token_acc={acc:.4f}", flush=True)
        if acc >= best_acc:
            best_acc = acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        save_checkpoint(model, optimizer, epoch, best_acc, best_state)
        if acc >= TARGET_ACC:
            print("early stop: target accuracy reached", flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), OUT_DIR / "agent-model.pt")
    size = (OUT_DIR / "agent-model.pt").stat().st_size
    metrics = {
        "rows": len(encoded),
        "parameters": int(params),
        "model_bytes": int(size),
        "val_token_accuracy": best_acc,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
    }
    (OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=1), encoding="utf-8")
    print(json.dumps(metrics, indent=1), flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/agent/export_agent_onnx.py
"""Export the trained 300M model to ONNX, then INT8-quantize to ~300MB.

SDPA has no direct ONNX lowering on older exporters, so the export temporarily
swaps in an eager matmul/softmax implementation (numerically identical).
"""
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn.functional as F

from model import AgentLM

HERE = Path(__file__).resolve().parent
RUNTIME = HERE / "runtime"
OUT = HERE / "agent-model.onnx"
OUT_INT8 = HERE / "agent-model-int8.onnx"


def eager_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):
    scale = query.size(-1) ** -0.5
    scores = torch.matmul(query, key.transpose(-2, -1)) * scale
    if attn_mask is not None:
        # Mirror PyTorch SDPA semantics exactly: bool True = participate,
        # float masks are added to the scores.
        if attn_mask.dtype == torch.bool:
            scores = scores.masked_fill(attn_mask.logical_not(), float("-inf"))
        else:
            scores = scores + attn_mask
    probs = torch.softmax(scores, dim=-1)
    if dropout_p > 0:
        probs = F.dropout(probs, dropout_p)
    return torch.matmul(probs, value)


def main() -> int:
    state_path = RUNTIME / "agent-model.pt"
    if not state_path.exists():
        raise SystemExit(f"missing trained weights: {state_path}")

    model = AgentLM()
    model.load_state_dict(torch.load(state_path, map_location="cpu"))
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)

    class Wrapper(torch.nn.Module):
        def __init__(self, inner: torch.nn.Module) -> None:
            super().__init__()
            self.inner = inner

        def forward(self, tokens: torch.Tensor) -> torch.Tensor:
            with torch.no_grad():
                return self.inner(tokens)

    original_sdpa = F.scaled_dot_product_attention
    F.scaled_dot_product_attention = eager_sdpa
    try:
        dummy = torch.randint(0, 256, (1, 512), dtype=torch.long)
        export_kwargs = dict(
            input_names=["tokens"],
            output_names=["logits"],
            dynamic_axes={"tokens": {1: "seq"}, "logits": {1: "seq"}},
            opset_version=17,
            do_constant_folding=True,
        )
        try:
            torch.onnx.export(Wrapper(model), (dummy,), str(OUT), dynamo=False, **export_kwargs)
        except TypeError:
            torch.onnx.export(Wrapper(model), (dummy,), str(OUT), **export_kwargs)
    finally:
        F.scaled_dot_product_attention = original_sdpa
    print("fp32 written", OUT, round(OUT.stat().st_size / 1073741824, 2), "GB")

    import numpy as np
    import onnxruntime

    session = onnxruntime.InferenceSession(str(OUT), providers=["CPUExecutionProvider"])
    rng = np.random.default_rng(3)
    probe = rng.integers(0, 256, size=(1, 64), dtype=np.int64)
    mutated = probe.copy()
    mutated[0, 32:] = rng.integers(0, 256, size=(32,), dtype=np.int64)
    early_a = session.run(None, {"tokens": probe})[0][0, :32]
    early_b = session.run(None, {"tokens": mutated})[0][0, :32]
    leak = float(np.max(np.abs(early_a - early_b)))
    print("causality check early-diff:", leak, flush=True)
    if leak > 1e-4:
        raise SystemExit("exported model is not causal; refuse to quantize")

    try:
        import gc

        # Free the torch weights before quantization so the two peaks do not stack.
        del model
        gc.collect()

        from onnxruntime.quantization import QuantType, quantize_dynamic

        quantize_dynamic(model_input=str(OUT), model_output=str(OUT_INT8), weight_type=QuantType.QInt8)
        print("int8 written", OUT_INT8, round(OUT_INT8.stat().st_size / 1048576, 1), "MB")
    except Exception as exception:  # noqa: BLE001 - quantization is optional
        print("int8 quantization skipped:", exception)

    try:
        import numpy as np
        import onnxruntime

        target = OUT_INT8 if OUT_INT8.exists() else OUT
        options = onnxruntime.SessionOptions()
        options.intra_op_num_threads = 4
        session = onnxruntime.InferenceSession(str(target), sess_options=options,
                                               providers=["CPUExecutionProvider"])
        ids = np.asarray([torch.randint(0, 256, (512,)).numpy()], dtype=np.int64)
        outputs = session.run(None, {"tokens": ids})
        print("onnx smoke ok:", target.name, outputs[0].shape)
    except Exception as exception:  # noqa: BLE001 - smoke is advisory
        print("onnx smoke skipped:", exception)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/agent/agent_model_runtime.py
"""ONNX decision-model runtime.

The model is optional: without it the daemon still runs the rule policy for
scheduling, but remediation actions are never emitted (safe default).
"""

import json
import os
import sys


class AgentModelRuntime:
    def __init__(self, model_path: str | None = None):
        here = os.path.dirname(os.path.abspath(__file__))
        self.path = model_path or os.path.join(here, "agent-model.onnx")
        self.session = None
        self.loaded = False
        self._input_name = None
        self._output_names = None
        self._max_context = 1024
        self._load()

    def _load(self) -> None:
        if not os.path.exists(self.path):
            print(f"[model] not found: {self.path}", file=sys.stderr)
            return
        try:
            import numpy  # noqa: F401  (onnxruntime needs numpy)
            import onnxruntime

            options = onnxruntime.SessionOptions()
            options.intra_op_num_threads = 1
            options.log_severity_level = 3
            self.session = onnxruntime.InferenceSession(
                self.path, sess_options=options, providers=["CPUExecutionProvider"]
            )
            self._input_name = self.session.get_inputs()[0].name
            self._output_names = [output.name for output in self.session.get_outputs()]
            self.loaded = True
            print(f"[model] loaded {os.path.basename(self.path)}", file=sys.stderr)
        except Exception as exception:  # noqa: BLE001 - optional component
            print(f"[model] load failed: {exception}", file=sys.stderr)
            self.session = None
            self.loaded = False

    def confirm_remediation(self, context: dict, action: dict) -> bool:
        """Return True only when the model explicitly accepts the action.

        Any failure (model missing, malformed output, timeout) returns False so
        the daemon errs on the side of not remediating.
        """
        if not self.loaded or self.session is None:
            return False
        try:
            prompt = self._build_confirm_prompt(context, action)
            text = self._generate(prompt, max_new=8)
            decision = text.strip().lower()
            if decision.startswith("a"):
                return True
            print(f"[model] remediation rejected: {text[:120]!r}", file=sys.stderr)
            return False
        except Exception as exception:  # noqa: BLE001 - confirmation is best effort
            print(f"[model] confirm failed: {exception}", file=sys.stderr)
            return False

    def _build_confirm_prompt(self, context: dict, action: dict) -> str:
        state = json.dumps(self._confirm_state(context, action), ensure_ascii=False,
                           separators=(",", ":"))[:1500]
        candidate = json.dumps(self._confirm_action(action), ensure_ascii=False,
                               separators=(",", ":"))[:800]
        return (
            "你是 NDAS 本地安全决策模型。判断下面的处置动作是否有充分证据。"
            '只回答 accept 或 reject。\nSTATE:' + state + "\nACTION:" + candidate + "\n"
        )

    @staticmethod
    def _confirm_state(context: dict, action: dict) -> dict:
        """Shape the live decision into the training distribution (flat fields)."""
        event = context.get("event") if isinstance(context, dict) else None
        event = event if isinstance(event, dict) else {}
        detections = event.get("detections")
        sources = []
        if isinstance(detections, list):
            for detection in detections:
                if isinstance(detection, dict) and detection.get("source"):
                    sources.append(str(detection["source"]))
        path = str(event.get("path") or "")
        file_name = path.rsplit("\\", 1)[-1].rsplit("/", 1)[-1] or "unknown"
        name = action.get("action") or ""
        return {
            "file": file_name,
            "kind": str(event.get("kind") or "pe"),
            "verdict": str(event.get("verdict") or "malicious"),
            "score": int(event.get("score") or 0),
            "evidence": sources,
            "case": "root" if (event.get("caseId") or context.get("caseId")) else "none",
            "mode": str(context.get("mode") or "auto"),
            "processRunning": bool(action.get("pid")),
            "p7bTrust": str(event.get("p7bTrust") or "none"),
        }

    @staticmethod
    def _confirm_action(action: dict) -> dict:
        name = action.get("action") or ""
        shaped = {"action": name, "reason": action.get("reason") or ""}
        args = action.get("args") if isinstance(action.get("args"), dict) else {}
        if name == "block_network" and args.get("domain"):
            shaped["domain"] = args["domain"]
        if name == "p7b_decide" and args.get("decision"):
            shaped["decision"] = args["decision"]
        return shaped

    def _generate(self, prompt: str, max_new: int = 48) -> str:
        if self.session is None:
            return ""
        import numpy

        # The exported graph hard-codes the encoder reshape to 512 positions, so
        # every step pads to exactly 512 (right padding matches the trainer).
        fixed = 512
        encoded = list(prompt.encode("utf-8"))[: fixed - 1]
        tokens = [256] + encoded
        generated: list[int] = []
        for _ in range(max_new):
            window = tokens[-fixed:]
            real_length = len(window)
            seq = window + [258] * (fixed - real_length)
            inputs = {self._input_name: numpy.asarray([seq], dtype=numpy.int64)}
            outputs = self.session.run(self._output_names, inputs)
            last = outputs[0][0, real_length - 1]
            next_token = int(last.argmax())
            if next_token in (256, 258, 257):
                break
            generated.append(next_token)
            tokens.append(next_token)
        return bytes(t for t in generated if 0 <= t <= 255).decode("utf-8", errors="replace")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['NDAS_CKPT_DIR'] = '/content/drive/MyDrive/ndas-agent'
os.makedirs(os.environ['NDAS_CKPT_DIR'], exist_ok=True)
print('checkpoints ->', os.environ['NDAS_CKPT_DIR'])

In [ ]:
%cd /content/agent
!python make_agent_data.py
!python train_agent_model.py

In [ ]:
%cd /content/agent
!python train_agent_model.py --resume

In [ ]:
%cd /content/agent
!python export_agent_onnx.py

In [ ]:
import os, shutil
candidates = ['/content/agent/agent-model-int8.onnx', '/content/agent/agent-model.onnx']
src = next((p for p in candidates if os.path.exists(p)), None)
if src is None:
    print('no exported model found; run the export cell first')
else:
    dst = '/content/drive/MyDrive/ndas-agent/' + os.path.basename(src)
    shutil.copy(src, dst)
    print('saved to Drive:', dst, '| size MB:', round(os.path.getsize(src)/1048576, 1))
    from google.colab import files
    files.download(src)